In [70]:
import pandas as pd 
import numpy as np


In [71]:
df=pd.read_csv('gurgaon_properties_post_feature_selection.csv')

In [72]:
df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servent room,store room,furnishing_type,luxury_category,floor_category,price
0,0.0,90.0,1.0,1,1.0,3.0,336.0,0,0,0,2.0,1.0,0.21
1,0.0,56.0,4.0,4,4.0,0.0,5350.0,0,0,0,2.0,1.0,5.90
2,0.0,0.0,3.0,3,4.0,0.0,1900.0,0,0,0,1.0,2.0,0.90
3,1.0,18.0,5.0,5,2.0,2.0,4518.0,0,0,0,1.0,1.0,10.00
4,0.0,44.0,2.0,2,4.0,3.0,1165.0,0,0,0,0.0,0.0,0.72


In [73]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder 
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from  sklearn.model_selection import KFold,cross_val_score,train_test_split

In [74]:
X=df.drop(columns='price')
y=df['price']

In [75]:
columns_to_encode=['sector','balcony','agePossession','furnishing_type','luxury_category','floor_category']

In [76]:
y_transform=np.log1p(y)

In [77]:
preprocessor=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),['property_type', 'bedRoom', 'bathroom', 'built_up_area', 'servent room', 'store room']),
        ('cat',OneHotEncoder(drop='first'),columns_to_encode)
    ],
    remainder='passthrough'
)

In [78]:
pipeline=Pipeline([
    ("preprocessing",preprocessor),
    ("regressor",LinearRegression())
    
])

In [79]:
K_fold=KFold(shuffle=True,n_splits=10,random_state=42)
score=cross_val_score(pipeline,X,y_transform,cv=K_fold,scoring='r2')

In [80]:
score

array([0.84208931, 0.82049523, 0.83861985, 0.84448593, 0.85590891,
       0.83700599, 0.78419269, 0.85195464, 0.86831766, 0.86423755])

In [81]:
score.mean()

np.float64(0.8407307757330054)

In [82]:
score.std()

np.float64(0.02304401144946488)

In [83]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transform,test_size=0.2,random_state=42)
pipeline.fit(X_train,y_train)

,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [84]:
y_pred=pipeline.predict(X_test)

In [85]:
y_pred

array([0.86941351, 1.33080455, 1.0063595 , 2.0295334 , 2.17198416,
       0.3081137 , 0.91477793, 0.89373031, 0.9849823 , 0.54448699,
       1.16204361, 1.27989412, 0.35708017, 1.57642449, 0.74220045,
       0.83151042, 0.64656845, 0.52023197, 0.96102577, 2.23803523,
       1.38321522, 0.71302133, 0.57693255, 0.43046118, 0.50130151,
       1.03894404, 0.75254617, 0.58910241, 1.35949204, 0.96061721,
       1.03190816, 0.71380491, 1.04534381, 1.37224904, 1.32030786,
       2.42986067, 0.84095537, 0.64449601, 0.52641028, 0.63203243,
       0.46286468, 0.86683762, 0.71497855, 1.12176568, 0.32542111,
       0.6784703 , 1.52605742, 1.18053356, 0.73223718, 1.90527394,
       1.97679011, 2.05380072, 1.23238494, 0.69276674, 0.88331613,
       0.69700807, 0.48394952, 0.88331613, 0.85029872, 0.44626116,
       2.5465126 , 1.17772897, 0.79431986, 1.42650946, 0.71655085,
       0.62664712, 0.75536908, 2.67805343, 1.32899752, 0.57916191,
       1.01772413, 0.68952553, 0.64012035, 1.23723507, 0.82344

In [86]:
y_pred=np.expm1(y_pred)

In [87]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

0.6343873696698124

In [88]:
from sklearn.svm import SVR

In [89]:
from sklearn.compose import ColumnTransformer
# X['built_up_area']=round(np.log1p(X['built_up_area'])) it reduse the output
preprocessor=ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),['property_type', 'bedRoom', 'bathroom', 'built_up_area', 'servent room', 'store room']),
        ('cat',OneHotEncoder(drop='first'),columns_to_encode)
    ]
)

In [90]:
pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('regresor',SVR(kernel='rbf'))
])

In [91]:
np.log1p(X['built_up_area'])

0       5.820083
1       8.585039
2       7.550135
3       8.416046
4       7.061334
          ...   
3634    7.854381
3635    6.952729
3636    7.755339
3637    7.427739
3638    7.576097
Name: built_up_area, Length: 3639, dtype: float64

In [92]:
K_fold=KFold(shuffle=True,n_splits=10,random_state=42)
score=cross_val_score(pipeline,X,y_transform,cv=K_fold,scoring='r2')
score

array([0.87730298, 0.87373095, 0.85914888, 0.85541835, 0.88439826,
       0.89986727, 0.8767424 , 0.87778128, 0.87773788, 0.90194422])

In [93]:
score.mean()

np.float64(0.8784072471724255)

In [94]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transform,test_size=0.2,random_state=42)
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regresor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [95]:
y_pred=pipeline.predict(X_test)

In [96]:
y_pred=np.expm1(y_pred)

In [97]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.5462197423678754